In [1]:
# ==========================================
# CELL 1: DATA PREPARATION (3-CLASS FUSION)
# ==========================================
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler

PROJECT_PATH = "Thesis_Data"
LABELS_PATH = "videos_with_sentiment_labels.csv"

df = pd.read_csv(LABELS_PATH)

visual_dict = np.load(f"{PROJECT_PATH}/visual_features_clip.npy", allow_pickle=True).item()
audio_dict = np.load(f"{PROJECT_PATH}/audio_features_vggish.npy", allow_pickle=True).item()

X_v_list, X_a_list, y_labels = [], [], []

for index, row in df.iterrows():
    v_id = row['video_id']
    # Using the original 3-class label now!
    label = row['majority_sentiment'] 
    
    if v_id in visual_dict and v_id in audio_dict:
        X_v_list.append(visual_dict[v_id])
        X_a_list.append(audio_dict[v_id])
        y_labels.append(label)

X_visual = np.array(X_v_list)
X_audio = np.array(X_a_list)
y_labels = np.array(y_labels)

# Apply Max Pooling (If your data is 3D)
def apply_max_pooling(features):
    if features.ndim == 3:
        return np.max(features, axis=1) 
    return features

X_visual_pooled = apply_max_pooling(X_visual)
X_audio_pooled = apply_max_pooling(X_audio)

# Concatenate visual and audio features
X_fused = np.concatenate((X_visual_pooled, X_audio_pooled), axis=1)

le = LabelEncoder()
y_encoded = le.fit_transform(y_labels)

print("Data Loaded! 3-Class Distribution:")
unique, counts = np.unique(y_encoded, return_counts=True)
for i in range(len(unique)):
    print(f"{le.classes_[i]}: {counts[i]} videos")

print(f"\nFused Feature Shape: {X_fused.shape} (Videos, Features)")

# Setup strict 5-Fold Cross-Validation
k_folds = 5
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

Data Loaded! 3-Class Distribution:
Negative: 52 videos
Neutral: 136 videos
Positive: 258 videos

Fused Feature Shape: (446, 640) (Videos, Features)


In [2]:
# ==========================================
# CELL 2: RANDOM FOREST (3-CLASS)
# ==========================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

rf_true, rf_preds = [], []
rf_accs = []

print("Starting Random Forest 5-Fold CV...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_fused, y_encoded)):
    X_train, X_val = X_fused[train_idx], X_fused[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    # Random Forest setup for Multi-class
    rf_model = RandomForestClassifier(
        n_estimators=200, 
        class_weight='balanced', 
        random_state=42, 
        n_jobs=-1
    )
    
    rf_model.fit(X_train_scaled, y_train)
    rf_fold_preds = rf_model.predict(X_val_scaled)
    
    rf_accs.append(accuracy_score(y_val, rf_fold_preds))
    rf_true.extend(y_val)
    rf_preds.extend(rf_fold_preds)

print("\n" + "="*50)
print("=== RANDOM FOREST MULTI-CLASS RESULTS ===")
print("="*50)
print(f"Average Accuracy: {np.mean(rf_accs):.4f} (+/- {np.std(rf_accs):.4f})\n")
print(classification_report(rf_true, rf_preds, target_names=le.classes_))

Starting Random Forest 5-Fold CV...

=== RANDOM FOREST MULTI-CLASS RESULTS ===
Average Accuracy: 0.5942 (+/- 0.0076)

              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00        52
     Neutral       0.56      0.13      0.21       136
    Positive       0.60      0.96      0.74       258

    accuracy                           0.59       446
   macro avg       0.39      0.36      0.32       446
weighted avg       0.52      0.59      0.49       446



In [4]:
# ==========================================
# CELL 3: XGBOOST (3-CLASS)
# ==========================================
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

xgb_true, xgb_preds = [], []
xgb_accs = []

print("Starting XGBoost 5-Fold CV...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_fused, y_encoded)):
    X_train, X_val = X_fused[train_idx], X_fused[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    # Calculate sample weights to handle 3-class imbalance
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
    
    # XGBoost setup for Multi-class
    xgb_model = XGBClassifier(
        n_estimators=200, 
        objective='multi:softprob', 
        num_class=len(le.classes_),
        eval_metric='mlogloss',
        random_state=42, 
        n_jobs=-1
    )
    
    # Feed the sample weights directly into the fit function
    xgb_model.fit(X_train_scaled, y_train, sample_weight=sample_weights)
    xgb_fold_preds = xgb_model.predict(X_val_scaled)
    
    xgb_accs.append(accuracy_score(y_val, xgb_fold_preds))
    xgb_true.extend(y_val)
    xgb_preds.extend(xgb_fold_preds)

print("\n" + "="*50)
print("=== XGBOOST MULTI-CLASS RESULTS ===")
print("="*50)
print(f"Average Accuracy: {np.mean(xgb_accs):.4f} (+/- {np.std(xgb_accs):.4f})\n")
print(classification_report(xgb_true, xgb_preds, target_names=le.classes_))

Starting XGBoost 5-Fold CV...

=== XGBOOST MULTI-CLASS RESULTS ===
Average Accuracy: 0.5873 (+/- 0.0567)

              precision    recall  f1-score   support

    Negative       0.38      0.06      0.10        52
     Neutral       0.46      0.35      0.39       136
    Positive       0.63      0.82      0.71       258

    accuracy                           0.59       446
   macro avg       0.49      0.41      0.40       446
weighted avg       0.55      0.59      0.55       446



In [5]:
# ==========================================
# CELL 4: BAYESIAN NEURAL NETWORK (3-CLASS)
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
import torchbnn as bnn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler

print("Starting Bayesian Neural Network 5-Fold CV...")

# 1. DEFINE THE BAYESIAN MODEL
class BayesianMLP(nn.Module):
    def __init__(self, input_dim=640, num_classes=3):
        super(BayesianMLP, self).__init__()
        # Instead of fixed weights, these layers hold probability distributions
        self.fc1 = bnn.BayesLinear(prior_mu=0, prior_sigma=0.1, in_features=input_dim, out_features=128)
        self.relu = nn.ReLU()
        self.fc2 = bnn.BayesLinear(prior_mu=0, prior_sigma=0.1, in_features=128, out_features=num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# Setup Storage
bnn_true, bnn_preds = [], []
bnn_accs = []
MAX_EPOCHS = 100

# 2. TRAINING LOOP
for fold, (train_idx, val_idx) in enumerate(skf.split(X_fused, y_encoded)):
    
    X_train, X_val = X_fused[train_idx], X_fused[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    train_loader = DataLoader(TensorDataset(
        torch.tensor(X_train_scaled, dtype=torch.float32), 
        torch.tensor(y_train, dtype=torch.long)
    ), batch_size=32, shuffle=True)
    
    val_loader = DataLoader(TensorDataset(
        torch.tensor(X_val_scaled, dtype=torch.float32), 
        torch.tensor(y_val, dtype=torch.long)
    ), batch_size=32, shuffle=False)
    
    model = BayesianMLP(input_dim=X_fused.shape[1], num_classes=len(le.classes_))
    
    # BNNs require TWO loss functions
    ce_loss_fn = nn.CrossEntropyLoss()
    kl_loss_fn = bnn.BKLLoss(reduction='mean', last_layer_only=False)
    
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    kl_weight = 0.01 # How much we penalize the network for being too "certain"
    
    # Train the model
    model.train()
    for epoch in range(MAX_EPOCHS):
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            
            # Combine standard loss with Bayesian KL loss
            ce_loss = ce_loss_fn(outputs, labels)
            kl_loss = kl_loss_fn(model)
            total_loss = ce_loss + (kl_weight * kl_loss)
            
            total_loss.backward()
            optimizer.step()
            
    # 3. BAYESIAN EVALUATION (The Monte Carlo Method)
    # Because BNN weights are random, we run the test set 10 times 
    # and average the predictions to get a stable answer!
    model.eval()
    fold_preds, fold_labels = [], []
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            # Run the same inputs through the network 10 times
            mc_predictions = []
            for _ in range(10): 
                outputs = model(inputs)
                mc_predictions.append(outputs.unsqueeze(0))
            
            # Stack all 10 runs and calculate the average prediction
            mc_predictions = torch.cat(mc_predictions, dim=0) # Shape: (10, batch_size, classes)
            mean_predictions = torch.mean(mc_predictions, dim=0) # Shape: (batch_size, classes)
            
            _, preds = torch.max(mean_predictions, 1)
            fold_preds.extend(preds.numpy())
            fold_labels.extend(labels.numpy())
            
    fold_acc = accuracy_score(fold_labels, fold_preds)
    bnn_accs.append(fold_acc)
    bnn_true.extend(fold_labels)
    bnn_preds.extend(fold_preds)

print("\n" + "="*50)
print("=== BAYESIAN NEURAL NETWORK (BNN) RESULTS ===")
print("="*50)
print(f"Average Accuracy: {np.mean(bnn_accs):.4f} (+/- {np.std(bnn_accs):.4f})\n")
print(classification_report(bnn_true, bnn_preds, target_names=le.classes_))

Starting Bayesian Neural Network 5-Fold CV...

=== BAYESIAN NEURAL NETWORK (BNN) RESULTS ===
Average Accuracy: 0.5628 (+/- 0.0508)

              precision    recall  f1-score   support

    Negative       0.23      0.13      0.17        52
     Neutral       0.45      0.43      0.44       136
    Positive       0.65      0.72      0.68       258

    accuracy                           0.56       446
   macro avg       0.44      0.43      0.43       446
weighted avg       0.54      0.56      0.55       446

